## Access this Notebook
You can launch this notebook in the US GHG Center JupyterHub by clicking the link below. If you are a new user, you should first sign up for the hub by filling out this [**request form**](https://docs.google.com/forms/d/e/1FAIpQLSdai8otCdrVQzJgev8mjDhzKyCg7jcrB3UeTXNHoCiaMKrkaQ/viewform) and providing the required information. 


Access the [**OCO-3 Snapshot Area Maps of Column CO₂ Concentrations**](https://us-ghg-center.github.io/ghgc-docs/user_data_notebooks/oco3-co2-sams-daygrid-v11r_User_Notebook.html) notebook in the US GHG Center JupyterHub. 

## Table of Contents
- [Data Summary and Application](#data-summary-and-application)
- [Approach](#approach)
- [About the Data](#about-the-data)
- [Install the Required Libraries](#install-the-required-libraries)
- [Query the STAC API](#query-the-stac-api)
- [Visual Comparison Across Time Periods](#visual-comparison-across-time-periods)
- [Map Out Selected Tiles](#map-out-selected-tiles)
- [Summary](#summary)

## Data Summary and Application
- **Spatial coverage**: Global, 52°N to 52°S latitude
- **Spatial resolution**: 2.25 km x 1.29 km
- **Temporal extent**: August 06, 2019 - Ongoing
- **Temporal resolution**: Daily
- **Unit**: Parts per million

For more, visit the [OCO-3 Snapshot Area Maps of Column CO₂ Concentrations](https://earth.gov/ghgcenter/data-catalog/oco3-co2-sams-daygrid-v11r) data overview page.  

## Approach

1. Identify available dates and temporal frequency of observations for the given collection using the GHGC API `/stac` endpoint. 
2. Pass the STAC item into the raster API `/collections/{collection_id}/items/{item_id}/tilejson.json` endpoint.
3. Using `folium.Map`, visualize the plumes.
4. After the visualization, perform zonal statistics for a given polygon.

   

## About the Data
#### OCO-3 Snapshot Area Maps of Column CO₂ Concentrations
OCO-3, mounted on the ISS, measures CO₂ and solar-induced chlorophyll fluorescence (SIF) to study carbon dynamics across Earth. Its unique Snapshot Area Mapping (SAM) mode enables high-resolution scans of localized targets like cities and volcanoes, with over 22,000 SAMs collected from Aug 2019 to Nov 2023. After a brief storage period, it resumed operations on July 16, 2024.
For more information regarding this dataset, please visit the [OCO-3 Snapshot Area Maps of Column CO₂ Concentrations](https://earth.gov/ghgcenter/data-catalog/oco3-co2-sams-daygrid-v11r) data overview page.

## Terminology
Navigating data via the GHGC API, you will encounter terminology that is different from browsing in a typical filesystem. We'll define some terms here which are used throughout this notebook.
- `catalog`:    All datasets available at the `/stac` endpoint
- `collection`: A specific dataset, e.g. OCO-3 Snapshot Area Maps of Column CO₂ Concentrations
- `item`:       One granule in the dataset
- `asset`:      A variable available within the granule, e.g. CH4 plume emissions
- `STAC API`:   **Sp**atio**T**emporal **A**sset **C**atalogs - Endpoint for fetching metadata about available datasets
- `Raster API`: Endpoint for fetching data itself, for imagery and statistics

# Install the Required Libraries
Required libraries are pre-installed on the GHG Center Hub, except the `tabulate` and `seaborn` libraries. If you need to run this notebook elsewhere, please install the libraries by running the following command line:

%pip install requests folium rasterstats pystac_client pandas matplotlib --quiet

In [1]:
# Import the following libraries
# For fetching from the Raster API
import requests
# For making maps
import folium
import folium.plugins
from folium import Map, TileLayer
# For talking to the STAC API
from pystac_client import Client
# For working with data
import pandas as pd
# For making time series
import matplotlib.pyplot as plt
# For formatting date/time data
import datetime
# Custom functions for working with GHGC data via the API
import ghgc_utils

# Query the STAC API

**STAC API Collection Names**

Now, you must fetch the dataset from the [**STAC API**](https://earth.gov/ghgcenter/api/stac/) by defining its associated STAC API collection ID as a variable. 
The collection ID, also known as the **collection name**, for the OCO-3 Snapshot Area Maps of Column CO₂ Concentrations dataset is [**oco3-co2-sams-daygrid-v11r**](https://earth.gov/ghgcenter/api/stac/collections/oco3-co2-sams-daygrid-v11r)*

**You can find the collection name of any dataset on the GHGC data portal by navigating to the dataset landing page within the data catalog. The collection name is the last portion of the dataset landing page's URL, and is also listed in the pop-up box after clicking "ACCESS DATA."*

In [2]:
# Provide the STAC and RASTER API endpoints
# The endpoint is referring to a location within the API that executes a request on a data collection nesting on the server.

# The STAC API is a catalog of all the existing data collections that are stored in the GHG Center.
STAC_API_URL = "https://dev.ghg.center/api/stac"

# The RASTER API is used to fetch collections for visualization
RASTER_API_URL = "https://dev.ghg.center/api/raster"

# The collection name is used to fetch the dataset from the STAC API. First, we define the collection name as a variable
collection_name = "oco3-co2-sams-daygrid-v11r"

In [3]:
# Fetch the collection from the STAC API using the appropriate endpoint
# The 'pystac_client' library allows a HTTP request possible
catalog = Client.open(STAC_API_URL)
collection = catalog.get_collection(collection_name)

# Print the properties of the collection to the console
collection

<CollectionClient id=oco3-co2-sams-daygrid-v11r>

Examining the contents of our `collection` under the `temporal` variable, we note that data is available from Jan 2015 to August 2025.

In [4]:
items = list(collection.get_items())  # Convert the iterator to a list
print(f"Found {len(items)} items")

Found 18379 items


In [12]:
# The search function lets you search for items within a specific date/time range
search = catalog.search(
    collections=collection_name,
    datetime=['2024-08-25T00:00:00Z','2024-08-26T00:00:00Z']
)
# Take a look at the items we found
print(f"# items in date range: {len(search.item_collection())}")
for item in search.item_collection():
    print(item)

# items in date range: 42
<Item id=oco3-co2_volcano0076_2024-08-26T170125Z_filtered_xco2>
<Item id=oco3-co2_volcano0017_2024-08-26T165919Z_filtered_xco2>
<Item id=oco3-co2_volcano0008_2024-08-26T184116Z_filtered_xco2>
<Item id=oco3-co2_volcano0004_2024-08-26T043359Z_filtered_xco2>
<Item id=oco3-co2_tccon124_2024-08-26T133046Z_filtered_xco2>
<Item id=oco3-co2_tccon122_2024-08-26T071936Z_filtered_xco2>
<Item id=oco3-co2_sif_atto_2_2024-08-26T152157Z_filtered_xco2>
<Item id=oco3-co2_fossil0236_2024-08-26T072639Z_filtered_xco2>
<Item id=oco3-co2_fossil0231_2024-08-26T150506Z_filtered_xco2>
<Item id=oco3-co2_fossil0156_2024-08-26T055048Z_filtered_xco2>
<Item id=oco3-co2_fossil0127_2024-08-26T072937Z_filtered_xco2>
<Item id=oco3-co2_fossil0117_2024-08-26T055245Z_filtered_xco2>
<Item id=oco3-co2_fossil0110_2024-08-26T164007Z_filtered_xco2>
<Item id=oco3-co2_fossil0106_2024-08-26T151814Z_filtered_xco2>
<Item id=oco3-co2_fossil0058_2024-08-26T072353Z_filtered_xco2>
<Item id=oco3-co2_fossil0055_

In [13]:
# Examine the first item in the collection
# Keep in mind that a list starts from 0, 1, 2... therefore items[0] is referring to the first item in the list/collection
search_items = search.item_collection()
search_items[0]

<Item id=oco3-co2_volcano0076_2024-08-26T170125Z_filtered_xco2>

In [14]:
for item in search_items:
    print(item.properties["start_datetime"])

2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-26T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z
2024-08-25T00:00:00Z


For this reason, we will use the item `id` to reference the items specifically. Let's make a dictionary where we can do this more efficiently:

In [15]:
items_dict = {"_".join(item.id.split("_")[1:3]): item for item in search_items}
# Print the keys of our dictionary, which we'll use to reference specific plumes
print(items_dict.keys())

dict_keys(['volcano0076_2024-08-26T170125Z', 'volcano0017_2024-08-26T165919Z', 'volcano0008_2024-08-26T184116Z', 'volcano0004_2024-08-26T043359Z', 'tccon124_2024-08-26T133046Z', 'tccon122_2024-08-26T071936Z', 'sif_atto', 'fossil0236_2024-08-26T072639Z', 'fossil0231_2024-08-26T150506Z', 'fossil0156_2024-08-26T055048Z', 'fossil0127_2024-08-26T072937Z', 'fossil0117_2024-08-26T055245Z', 'fossil0110_2024-08-26T164007Z', 'fossil0106_2024-08-26T151814Z', 'fossil0058_2024-08-26T072353Z', 'fossil0055_2024-08-26T011048Z', 'fossil0052_2024-08-26T024913Z', 'fossil0047_2024-08-26T085537Z', 'fossil0039_2024-08-26T164239Z', 'fossil0036_2024-08-26T164727Z', 'fossil0027_2024-08-26T133407Z', 'ecostress_us', 'coccon102_2024-08-26T105007Z', 'volcano0038_2024-08-25T081959Z', 'volcano0020_2024-08-25T095838Z', 'volcano0003_2024-08-25T081107Z', 'val007_Reims', 'tccon124_2024-08-25T141920Z', 'tccon107_2024-08-25T035017Z', 'fossil0232_2024-08-25T141629Z', 'fossil0222_2024-08-25T002926Z', 'fossil0166_2024-08-25T

In [18]:
# Before we go further, let's pick which asset to focus on for the remainder of the notebook. 
# This dataset only has one asset to choose from:
asset_name = "cog_default"

# Creating Maps using Folium
You will now explore global carbon dioxide emissions and visualize the results on a map using `folium`.

## Fetch Imagery from Raster API
Here we get information from the `Raster API` which we will add to our map in the next section.

In [19]:
# You can change the <datetime_plumenumber> key below to look at a different plume.
observation_date_1 = items_dict["volcano0076_2024-08-26T170125Z"]

# Extract collection name and item ID
collection_id = observation_date_1.collection_id
item_id = observation_date_1.id

In [20]:
object = observation_date_1.assets[asset_name]
raster_bands = object.extra_fields.get("raster:bands", [{}])
rescale_values = {
    "max": raster_bands[0].get("histogram", {}).get("max"),
    "min": 0,
}

print(rescale_values)
print(raster_bands)

{'max': 425.2371826171875, 'min': 0}
[{'scale': 1.0, 'nodata': 'nan', 'offset': 0.0, 'sampling': 'area', 'data_type': 'float32', 'histogram': {'max': 425.2371826171875, 'min': 417.9154968261719, 'count': 11, 'buckets': [99, 401, 1356, 3659, 6331, 8974, 8968, 3969, 1005, 206]}, 'statistics': {'mean': 421.9550732097918, 'stddev': 1.074915516618739, 'maximum': 425.2371826171875, 'minimum': 417.9154968261719, 'valid_percent': 5.46375}}]


Now, you will pass the `item id`, `collection name`, `asset name`, and the `rescale values` to the Raster API endpoint, along with a colormap. This step tells the Raster API which collection, item, and asset you want to view, specifying the colormap and colorbar ranges to use for visualization. The API returns a JSON with information about the requested image. Each image will be referred to as a tile.

In [24]:
# Choose a colormap for displaying the tiles
# Make sure that the capitalization matches Matplotlib standards
# For more information on Colormaps in Matplotlib, please visit https://matplotlib.org/stable/users/explain/colors/colormaps.html
color_map = "jet"

In [25]:
# Make a GET request to retrieve information for the date specified
observation_date_1_tile = requests.get(
    f"{RASTER_API_URL}/collections/{collection_id}/items/{item_id}/tilejson.json?"
    f"&assets={asset_name}"
    f"&color_formula=gamma+r+1.05&colormap_name={color_map.lower()}"
    f"&rescale=400,420"
).json()

# Print the properties of the retrieved granule to the console
observation_date_1_tile


{'tilejson': '2.2.0',
 'version': '1.0.0',
 'scheme': 'xyz',
 'tiles': ['https://dev.ghg.center/api/raster/collections/oco3-co2-sams-daygrid-v11r/items/oco3-co2_volcano0076_2024-08-26T170125Z_filtered_xco2/tiles/WebMercatorQuad/{z}/{x}/{y}@1x?assets=cog_default&color_formula=gamma+r+1.05&colormap_name=jet&rescale=400%2C420'],
 'minzoom': 0,
 'maxzoom': 24,
 'bounds': [-68.68187765185914,
  -23.771877804447026,
  -65.67812295849244,
  -20.768123111080317],
 'center': [-67.18000030517578, -22.270000457763672, 0]}

In [26]:
# Set initial zoom and center of map for plume Layer
# We'll use the "center" variable from our loaded tile to set the center of the map
# Note that we specify "tiles=None" because in the next step we're going to set a custom tile to serve as our underlying world map.
map_ = folium.Map(location=(observation_date_1_tile["center"][1], observation_date_1_tile["center"][0]), zoom_start=12, tiles=None, tooltip = 'test tool tip')
# Specify a custom imagery source for the underlying map
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}.png', name='ESRI World Imagery', attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',overlay='True').add_to(map_)
# Add place labels on top
folium.TileLayer(tiles='https://server.arcgisonline.com/arcgis/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}.png',name='ESRI World Boundaries and Places',attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',overlay='True').add_to(map_)

# Use the 'TileLayer' library display the raster layer
map_layer = TileLayer(
    tiles=observation_date_1_tile["tiles"][0], # Path to retrieve the tile
    name=f'{items[0].assets[asset_name].title}', # Give this layer a title
    overlay='True', # The layer can be overlaid on the map
    attr="GHG", # Set the attribution
    opacity=1, # Adjust the transparency of the layer
)
map_layer.add_to(map_)

# Adjust map elements 
folium.LayerControl(collapsed=False, position='topright').add_to(map_)

# Add colorbar
# We can use one of 'generate_html_colorbar' from the 'ghgc_utils' module 
# to create an HTML colorbar representation.
legend_html = ghgc_utils.generate_html_colorbar(
                color_map,
                rescale_values,
                label='XCO2 (ppm-m)'
    )

# Add colorbar to the map
map_.get_root().html.add_child(folium.Element(legend_html))


# Visualizing the map
map_

## Summary

In this notebook we have successfully completed the following steps for the STAC collection for the OCO-3 Snapshot Area Maps of Column CO₂ Concentrations dataset:
1.  Install and import the necessary libraries
2.  Fetch the collection from STAC collections using the appropriate endpoints
3.  Count the number of existing granules within the collection
4.  Map the Carbon Dioxide emissions

If you have any questions regarding this user notebook, please contact us using the [feedback form](https://docs.google.com/forms/d/e/1FAIpQLSeVWCrnca08Gt_qoWYjTo6gnj1BEGL4NCUC9VEiQnXA02gzVQ/viewform).